# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n\nLicense: {metadata.license}\n\nPublished: {metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s for this dataset.

**Note:** All schema components in Croissant (record sets, fields, columns) are referenced internally by `@id`.

Let's list all record sets, their IDs, and describe their fields.

In [ ]:
# List all available record sets in the dataset by @id and list fields
record_sets = list(dataset.record_sets.values())
print(f"Found {len(record_sets)} record sets:\n")
for rec_set in record_sets:
    print(f"RecordSet @id: {rec_set.id}")
    print(f"  Name: {rec_set.name}")
    print(f"  Description: {getattr(rec_set, 'description', 'No description')}\n  Fields:")
    if hasattr(rec_set, 'fields') and rec_set.fields:
        for fld in rec_set.fields.values():
            print(f"    Field @id: {fld.id} | name: {fld.name} | dataType: {fld.data_type}")
    print("-"*70)

For illustration, let's inspect the first record set by its `@id`.

In [ ]:
# Show a preview of records for the first available record set
if len(record_sets) > 0:
    chosen_record_set_id = record_sets[0].id
    print(f"Showing up to 2 records for record set: {chosen_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(x)
        if i >= 1:
            break
else:
    print('No record sets found in the schema!')

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

We'll use the record set and field `@id`s found in the previous section.

In [ ]:
# Extract data from each record set by @id
all_record_set_ids = [r.id for r in record_sets]

dataframes = {}
for rsid in all_record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

if all_record_set_ids:
    example_records_set_id = all_record_set_ids[0]
    print(f"Columns in DataFrame for record set {example_records_set_id}:")
    print(dataframes[example_records_set_id].columns.tolist())
    display(dataframes[example_records_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field from the chosen record set. We'll filter records, normalize a numeric field, and (if available) group by a categorical field.

First, preview column names:

In [ ]:
# Set up candidate numeric and categorical fields
df = dataframes[example_records_set_id]
print("DataFrame columns:", df.columns.tolist())

# Attempt to auto-detect a numeric field (float/int)
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_field_candidates:
    # Try to coerce columns to numeric
    numeric_field_candidates = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().all()]
print("Numeric field candidates:", numeric_field_candidates)

# Pick first numeric field for demo
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Selected numeric field: {numeric_field}\n")
else:
    print("No numeric fields found!")

# Attempt to pick a group field
group_field_candidates = [col for col in df.columns if df[col].nunique() < df.shape[0] // 3 and col != numeric_field]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    print(f"Selected group field: {group_field}\n")

In [ ]:
if numeric_field_candidates:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (mean): {len(filtered_df)} rows\n")

    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, col_norm]].head())

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by '{group_field}':\n", grouped_df.head())

## 5. Visualization

Let's plot a histogram of the numeric field and, if a group field is available, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=numeric_field, x=group_field, data=df)
        plt.title(f"Distribution of '{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, you explored the FAIR² dataset using the `mlcroissant` library. You loaded record sets by their Croissant `@id`, extracted tabular data, performed filtering and normalization, and visualized distribution of numeric fields. This approach can be extended to deeper domain-specific protein analysis, peptide quantitation, and more, using the explicit schema and metadata provided by Croissant.

Remember that all data access in Croissant is via entity `@id` for record sets and fields, ensuring stable reference and reproducibility.